# Knapsack with QAOA — Guided Tutorial (Part 01)

**SMU Quantum Optimisation Group · Japan–Singapore Workshop on Applied Quantum Optimisation**

⏱️ **~12 minutes** · run the cells top-to-bottom.

### What you'll learn
1. Turn a **knapsack** into a **QuadraticProgram → QUBO → Ising Hamiltonian**.
2. Build a **QAOA** circuit and optimise its angles on a **local simulator**.
3. **Sample** the final circuit and **decode** the best *feasible* solution.
4. Compare the quantum result against the **exact CPLEX optimum**.

### The pipeline (every notebook follows this)
`Problem → QuadraticProgram → QUBO (+slack) → Ising H → QAOA circuit → sample → decode`

### This instance
6 items · weights `[3,5,6,7,9,10]` · values `[5,10,20,30,35,40]` · capacity `14`.
**Exact optimum = value 50** (items 3 & 4, weight 13). QUBO uses **~10 qubits** (6 items + ~4 slack).

> ⚠️ **QAOA is heuristic.** Measurement returns *samples from a distribution*, not a guaranteed optimum — so we always keep the best **feasible** sample and compare to the classical benchmark.

---

### Knapsack Problem using Quantum Approximate Optimization Algorithm with Qiskit

This tutorial will walk you through solving the Knapsack problem, a classic optimization challenge, using the Quantum Approximate Optimisation Algorithm (QAOA). We will use Qiskit to define the problem, convert it into a format suitable for a quantum computer, and then use QAOA to find an approximate solution.

First, let's ensure Qiskit is installed and check the version.

In [ ]:
import qiskit
print(qiskit.__version__)

### The Knapsack Problem

The Knapsack problem involves selecting a set of items, each with a specific weight and value, to maximize the total value without exceeding a given maximum weight capacity. It's a binary optimization problem: for each item, you either include it in the knapsack or you don't.

### Setting Up the Environment

Before we define the problem, we need to import the necessary libraries. These include:

- **Basic libraries:** numpy for numerical operations and matplotlib for plotting.

- **Qiskit Optimization:** Tools to define the Knapsack problem (Knapsack), convert it into a quadratic program, and map it to a quantum problem (QuadraticProgramToQubo).

- **Qiskit Algorithms:** The core QAOA components, including the Estimator primitive for calculating expectation values and the EfficientSU2 ansatz.

- **SciPy:** A classical optimizer that QAOA will use to minimize the cost function.

- **Qiskit Aer:** To simulate a quantum backend for our experiment.

In [ ]:
# basic imports

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph

# quantum imports
from qiskit_optimization.algorithms import CplexOptimizer
from qiskit_optimization.applications import Maxcut, Knapsack
from qiskit.circuit import Parameter,QuantumCircuit
from qiskit_optimization.translators import from_docplex_mp
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector
# Pre-defined ansatz circuit and operator class for Hamiltonian
from qiskit.circuit.library import EfficientSU2
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit.circuit.library import QAOAAnsatz
# SciPy minimizer routine
from scipy.optimize import minimize
from qiskit.primitives import BackendEstimatorV2, BackendSamplerV2
from qiskit_aer import AerSimulator
backend = AerSimulator(method='automatic')


estimator = BackendEstimatorV2(backend=backend)
sampler = BackendSamplerV2(backend=backend)

### **Generate a Random Instance**

To make things interesting, let's create a random instance of the Knapsack problem. We'll define the number of items and generate random weights and values for each. The knapsack's capacity is set to 60% of the total weight of all items.

We then create a `Knapsack` object and convert it to a `QuadraticProgram`, which is a standard format for optimization problems.

> ▶ **Define the instance** — 👀 build the `Knapsack`, convert to a `QuadraticProgram`, and print it. Note the 6 binary variables.

In [ ]:
# Set the random seed for reproducibility
np.random.seed(135)

# Use the same instance as part02/custom-penalty.ipynb
num_items = 6
weights = np.array([3, 5, 6, 7, 9, 10])
values = np.array([5, 10, 20, 30, 35, 40])
capacity = 14

print(f"Weights: {weights}")
print(f"Values: {values}")
print(f"Capacity: {capacity}")

# Create the Knapsack problem
knapsack = Knapsack(values.tolist(), weights.tolist(), capacity)

# Convert the problem to a QuadraticProgram
problem = knapsack.to_quadratic_program()
print(problem.prettyprint())



In [ ]:
num_binary_vars_classical = problem.get_num_binary_vars()
print(f"Number of binary variables: {num_binary_vars_classical}")

### **Finding the Exact Solution Classically**

Before turning to a quantum approach, let's find the exact solution using a classical optimizer. This will serve as our benchmark to see how well QAOA performs. We use the `CplexOptimizer` from Qiskit Optimization for this purpose. The solution `x` is a binary vector where `1` means an item is selected and `0` means it's left out.

In [ ]:
# Solve the problem using CplexOptimizer
optimizer = CplexOptimizer()
result = optimizer.solve(problem)

result_classical = result
print(f"Optimal value: {result_classical.fval}, Solution vector: {result_classical.x}")


### **From Quadratic Program to QUBO**

Quantum computers solve problems formulated as Hamiltonians. A common step is to first convert the problem into a **Quadratic Unconstrained Binary Optimization (QUBO)** problem. This format represents the problem as a single quadratic equation to be minimized, without any constraints.

The conversion automatically creates "slack" variables to transform the inequality constraint (total weight <= capacity) into an equality, which is then incorporated into the objective function with a penalty.

> ▶ **Convert to QUBO** — 👀 slack variables appear here; the variable count jumps from **6 → ~10**.

In [ ]:
# problem to qubo
converter = QuadraticProgramToQubo()
qubo = converter.convert(problem)
print(qubo.export_as_lp_string())

As you can see, the QUBO has more variables than the original `num_items` because of the slack variables introduced to handle the weight constraint. Let's check the new number of variables.

In [ ]:
num_vars = qubo.get_num_vars()
print(f"Number of variables in QUBO: {num_vars}")
print(f"Number of variables in classical formulation: {num_binary_vars_classical}")
increase = num_vars - num_binary_vars_classical
percent_increase = (increase / num_binary_vars_classical) * 100
print(f"Number of variables increased by {increase} ({percent_increase:.1f}%) when converting to QUBO (due to slack variables for constraints).")

### **Mapping the QUBO to an Ising Hamiltonian**

QAOA works by preparing and sampling a parameterised state associated with a cost Hamiltonian. We now convert our QUBO problem into an **Ising Hamiltonian**. This Hamiltonian is an operator that can be measured on a quantum computer. Each binary variable in the QUBO is mapped to a qubit.

The conversion gives us two components:

- `qubitOp`: The Ising Hamiltonian, represented as a sum of Pauli operators (Z, ZZ).

- `offset`: A constant energy shift that we'll add back to our final result.

> ▶ **Map to an Ising Hamiltonian** — 👀 a sum of Pauli `Z` and `ZZ` terms, plus a constant `offset`.

In [ ]:
qubitOp, offset = qubo.to_ising()
print("Offset:", offset)
print("Ising Hamiltonian:")
print(str(qubitOp))

### **The Quantum Approximate Optimization Algorithm (QAOA)**

QAOA is another hybrid quantum-classical algorithm designed for optimization problems. It uses a special type of ansatz circuit that alternates between two operators: a **cost operator** (derived from our problem Hamiltonian) and a **mixer operator**.

The goal is to find the optimal parameters (angles) for these operators that prepare a quantum state with the lowest possible energy.



#### **The QAOA Ansatz**

The `QAOAAnsatz` circuit is constructed directly from our problem's cost operator (`qubitOp`). The `reps` parameter defines how many times the cost and mixer layers are repeated, which affects the circuit's complexity and potential accuracy.

Let's visualize the structure of our QAOA ansatz circuit with `reps=2 (the value we use below)`.

> ▶ **Build & draw the QAOA circuit** — 👀 alternating **cost** and **mixer** layers (`reps=2`).

In [ ]:
reps = 2
circuit = QAOAAnsatz(cost_operator=qubitOp, reps=reps)
circuit = circuit.decompose(reps=3)
circuit.draw('mpl',fold=-1)

#### Executing the QAOA Algorithm

To run QAOA, we need to set up the classical optimization loop.

- **Initial Parameters:** We choose an initial set of parameters (beta and gamma angles) for the ansatz. A common starting point is `pi` for gamma and `pi/2` for beta, repeated for each rep.

- **Cost Function:** We define a `cost_func_estimator` that takes the parameters, builds the QAOA circuit, and uses the `Estimator` to calculate the expectation value (energy) of the Hamiltonian. This energy is what the classical optimizer will minimize. We also store the objective function values at each step to plot the convergence.

- **Classical Optimizer:** We use the `COBYLA` optimizer from SciPy to find the optimal beta and gamma angles that minimize the energy.

Here are our initial parameters:

In [ ]:
initial_gamma = np.pi
initial_beta = np.pi/2

init_params = [initial_gamma, initial_beta]*reps
print(init_params)

This is the cost function that the optimizer will call:

In [ ]:
def cost_func_estimator(params, ansatz, hamiltonian, estimator):

    # transform the observable defined on virtual qubits to
    # an observable defined on all physical qubits
    isa_hamiltonian = hamiltonian.apply_layout(ansatz.layout)

    pub = (ansatz, isa_hamiltonian, params)
    job = estimator.run([pub])

    results = job.result()[0]
    cost = results.data.evs

    objective_func_vals.append(cost)


    return cost

Now, we execute the optimization loop.

> ▶ **Optimise the angles** (hybrid loop) — 👀 COBYLA lowers the energy over ~300 iterations. *This is the slow cell.*

In [ ]:
objective_func_vals = [] # Store the objective function values

result = minimize(
    cost_func_estimator,
    init_params,
    args= (circuit, qubitOp, estimator),
    method="cobyla", # you can also use 'SLSQP' or 'L-BFGS-B'
    options={'maxiter': 300},  # lowered for a live demo; raise to 1000 for tighter convergence, 'disp': True},
    tol=1e-8)
print(result)

The optimizer has found the optimal parameters (according to QAOA) . Let's plot the objective function value at each iteration to see how the algorithm converged to the solution.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(objective_func_vals, marker='o', linestyle='-', color='royalblue', label='Objective Value')
plt.title("QAOA Optimization Convergence", fontsize=16)
plt.xlabel("Iteration", fontsize=14)
plt.ylabel("Cost (Energy)", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()



### **Post-Processing: Sampling the Solution**

With the optimal parameters found, we now create the final QAOA circuit. We then add measurements and use the `Sampler` primitive to execute the circuit many times (shots) and obtain a probability distribution of the possible solution bitstrings.

The bitstring with the highest probability is our candidate for the optimal solution.

> ▶ **Sample the final circuit** — 👀 10,000 shots → a distribution over bitstrings.

In [ ]:
# Assign the optimal parameters to the ansatz
optimal_circuit = circuit.assign_parameters(result.x)
optimal_circuit.measure_all()

# Run the sampler job
pub = (optimal_circuit,)
job = sampler.run([pub], shots=int(1e4))

# Get the counts
counts_int = job.result()[0].data.meas.get_int_counts()
counts_bin = job.result()[0].data.meas.get_counts()

# Normalize the counts to get a probability distribution
shots = sum(counts_int.values())
final_distribution_int = {key: val/shots for key, val in counts_int.items()}
final_distribution_bin = {key: val/shots for key, val in counts_bin.items()}

Let's find the most likely bitstring from the final distribution.

> ▶ **Decode the best feasible answer** — 👀 we keep the highest-value *feasible* bitstring, not the most frequent raw one.

In [ ]:
# Select the highest-value feasible sample, rather than blindly using
# the most frequent raw bitstring (which may be infeasible).
def to_bitstring(integer, num_bits):
    result = np.binary_repr(integer, width=num_bits)
    return [int(digit) for digit in result]

keys = list(final_distribution_int.keys())
values = list(final_distribution_int.values())
feasible_candidates = []
for key, probability in final_distribution_int.items():
    candidate_bitstring = to_bitstring(key, num_vars)
    candidate_bitstring.reverse()
    candidate_solution = converter.interpret(candidate_bitstring)
    if problem.get_feasibility_info(candidate_solution)[0]:
        candidate_value = problem.objective.evaluate(candidate_solution)
        feasible_candidates.append((candidate_value, probability, candidate_bitstring))

if feasible_candidates:
    _, _, most_likely_bitstring = max(feasible_candidates, key=lambda row: (row[0], row[1]))
else:
    most_likely = keys[np.argmax(np.abs(values))]
    most_likely_bitstring = to_bitstring(most_likely, num_vars)
    most_likely_bitstring.reverse()

print("Result bitstring:", most_likely_bitstring)

In [ ]:
result = converter.interpret(most_likely_bitstring)
cost = problem.objective.evaluate(result)
feasible =problem.get_feasibility_info(result)[0]


print("Result knapsack:", result)
print("Result value:", cost)
print("Feasible:", feasible)

> ▶ **Compare quantum vs classical** — 👀 QAOA value vs the exact **50**, and the optimality gap.

In [ ]:
print("="*40)
print("🔎 Best Known Classical Solution")
print("="*40)
print(f"  • Total Value:        {result_classical.fval}")
print(f"  • Solution Vector:    {result_classical.x}")
print()

print("="*40)
print("⚛️  QAOA (Quantum) Solution")
print("="*40)
print(f"  • Total Value:        {cost}")
print(f"  • Solution Vector:    {result}")
print(f"  • Feasible:           {feasible}")

if feasible:
    optimality_gap = 100 * (result_classical.fval - cost) / result_classical.fval
    print(f"  • Optimality Gap:     {optimality_gap:.2f}%")
else:
    print("  • Note: Quantum solution is not feasible.")

print("\n" + "-"*40)
if feasible:
    if abs(cost - result_classical.fval) < 1e-6:
        print("✅ Quantum solution matches the classical optimum!")
    else:
        print("ℹ️  Quantum solution is suboptimal compared to classical.")
else:
    print("❌ Quantum solution is not feasible.")
print("-"*40)

---
### ✅ Recap
You modelled a knapsack, converted it to a QUBO/Ising form, ran **QAOA on a simulator**, and decoded a **feasible** solution that you compared to the exact optimum (**50**).

**Next:** open [`ibm_hardware_qaoa.ipynb`](ibm_hardware_qaoa.ipynb) to run the *same* circuit on a real IBM Quantum device.